# Chapter 29 — Failures, Retries, and Recovery

**Companion to *Pi Agents*** · [`Pi Agents` chapter 29](https://programmer.ie/books/pi/29-chapter/)

| | |
|---|---|
| Chapter | `29-chapter.md` · weight 29 · `/books/pi/29-chapter/` |
| Notebook | `notebooks/pi/29-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Does a normalised overflow error produce a compact-and-retry, and does it leave
a `context_edit` rather than deleting anything?

The chapter's point: Pi recognises a **specific** context-overflow signal. A
provider whose message Pi does not recognise needs a `message_end` handler that
normalises it. The recovery compacts, **omits the failed attempt with a
`context_edit`**, and runs again — it does not delete anything.


## What this notebook establishes

- A provider's own overflow message, **normalised** by a `message_end` handler, triggers Pi's compact-and-retry recovery as a fresh run.
- The recovery emits `compaction_start:overflow` and `compaction_end:overflow`, then runs again.
- The failed attempt is **omitted from context with a `context_edit`**, not deleted from the file — a `compaction` entry is also written.
- **Without the handler** the same message is just a failure: no recovery is attempted.
- A **rate limit that happens to contain 'length'** is left alone — Pi's classification is specific, not substring-based on a generic word.

## What this notebook does **not** establish

- **Retry timings and a provider's transient-error classification are not verified** (`ch29-lim1`).
- **The substring is a stand-in** for a real provider's message — the handler matches `model is too long`.
- **One compact-and-retry attempt**, not a loop, is Pi's documented recovery limit.
- **The handler only rewrites the error message** — it does not change the model, the run, or the retry policy.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — real sessions with the chapter's `message_end` handler, driven by a scripted model that returns an overflow error.
* **Evidence scope:** `in_process_runtime`. The classification, recovery and context_edit are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(29))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/29-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: a normalised overflow triggers compact-and-retry

In [3]:
FAIL = pinb.driver_source("ch29-failures.ts")
fail = pinb.driver(FAIL, label="ch29-failures", timeout=300)
n = fail["normalisedOverflow"]
print("Event sequence:", " -> ".join(n["events"]))
print()
pinb.show_checks([
    pinb.check("recovery compacted with reason overflow",
               n["compactionStartOverflow"], "compaction_start:overflow"),
    pinb.check("and then ran again (compaction_end before agent_end)",
               n["compactionEndBeforeAgentEnd"], "ran again"),
    pinb.check("the failed attempt was omitted with a context_edit, not deleted",
               n["hasContextEdit"], "context_edit written"),
    pinb.check("a compaction entry was written", n["hasCompaction"], "compaction written"),
    pinb.check("the run recovered", n["recovered"], "recovered after compaction"),
])

Event sequence: agent_end -> compaction_start:overflow -> compaction_end:overflow -> agent_end -> agent_settled

== Assertions ==
  assertion  observed
----  ---------------------------------------------------------------  --------------------------
PASS  recovery compacted with reason overflow  compaction_start:overflow
PASS  and then ran again (compaction_end before agent_end)  ran again
PASS  the failed attempt was omitted with a context_edit, not deleted  context_edit written
PASS  a compaction entry was written  compaction written
PASS  the run recovered  recovered after compaction

5/5 assertions held.


## Contrasting case: without the handler, no recovery

In [4]:
w = fail["withoutHandler"]
print("Event sequence without the handler:", " -> ".join(w["events"]))
print()
pinb.show_checks([
    pinb.check("without the handler, Pi does not recognise the message as an overflow",
               not w["compactionStarted"], "no compaction started"),
])

Event sequence without the handler: agent_end -> agent_settled

== Assertions ==
  assertion  observed
----  ---------------------------------------------------------------------  ---------------------
PASS  without the handler, Pi does not recognise the message as an overflow  no compaction started

1/1 assertions held.


## The classification is specific: a rate limit with 'length' is left alone

In [5]:
rl = fail["rateLimit"]
print("Event sequence for a rate limit:", " -> ".join(rl["events"]))
print()
pinb.show_checks([
    pinb.check("a rate limit containing the word length is not treated as an overflow",
               not rl["compactionStarted"], "no compaction started"),
])

Event sequence for a rate limit: agent_end -> agent_settled

== Assertions ==
  assertion  observed
----  ---------------------------------------------------------------------  ---------------------
PASS  a rate limit containing the word length is not treated as an overflow  no compaction started

1/1 assertions held.


## The chapter's own tests

All three failure tests.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [6]:
PATTERNS = 'ch29-failures/overflow.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch29-failures/overflow.test.ts', show="overflow")

$ node --test ch29-failures/overflow.test.ts
  PASS  3 passed, 0 failed, 0 skipped  in 968 ms

  tests matching 'overflow':
    ok   a provider's own overflow message, normalised, triggers compact-and-retry as a fresh run


## Your turn: predict, then run

**Predict first.** The handler matches `model is too long`. What happens if the
provider says `too many tokens` instead — does the handler normalise it?

**Then change one input.** Change the handler to match a broader string like
`too long`. What happens to a message that says `the file name is too long`?

**Predict the boundary.** Pi allows **one** compact-and-retry. What happens if the
retry also overflows — does it try again, or fail?

In [7]:
print("Predictions:")
print("  1. too many tokens: NOT normalised - the handler matches only model is too long.")
print("  2. Broader match: a file-name error would be misclassified as an overflow.")
print("  3. Retry overflows again: Pi fails - the recovery is one attempt, not a loop.")
print()
print("Observed above:")
print(f"  normalised: start={n['compactionStartOverflow']}, end-before-agent-end={n['compactionEndBeforeAgentEnd']}, recovered={n['recovered']}")
print(f"  context_edit written: {n['hasContextEdit']}, compaction written: {n['hasCompaction']}")
print(f"  without handler: compaction started={w['compactionStarted']}")
print(f"  rate limit with length: compaction started={rl['compactionStarted']}")
print()
assert n["compactionStartOverflow"] and n["compactionEndBeforeAgentEnd"]
assert n["hasContextEdit"] and n["hasCompaction"] and n["recovered"]
assert not w["compactionStarted"] and not rl["compactionStarted"]
print("held: normalised overflow recovers with a context_edit; without it, failure; the classification is specific")

Predictions:
  1. too many tokens: NOT normalised - the handler matches only model is too long.
  2. Broader match: a file-name error would be misclassified as an overflow.
  3. Retry overflows again: Pi fails - the recovery is one attempt, not a loop.

Observed above:
  normalised: start=True, end-before-agent-end=True, recovered=True
  context_edit written: True, compaction written: True
  without handler: compaction started=False
  rate limit with length: compaction started=False

held: normalised overflow recovers with a context_edit; without it, failure; the classification is specific


## Interpretation

Recovery is a **repair path with a hard limit**, and the chapter shows the
boundaries:

| Signal | What happens | The boundary |
|---|---|---|
| Pi-recognised overflow | Compact-and-retry | **One** attempt, not a loop |
| Normalised by handler | Compact-and-retry | The handler rewrites the message only |
| Unrecognised error | Plain failure | No recovery attempted |
| Rate limit with "length" | Plain failure | Classification is specific, not substring |

The failed attempt is **omitted** from context with a `context_edit`, and a
`compaction` entry is written. Nothing is deleted from the file — the same
property chapter 23 found for compaction.

Practical rules:
1. **Know your provider's overflow message.** If Pi does not recognise it, normalise it with `message_end`.
2. **Match precisely.** A broad substring match misclassifies unrelated errors.
3. **Recovery is one attempt.** A retry that also overflows fails.
4. **The failure is not deleted.** `context_edit` omits it from context; the file keeps it.
5. **`session_compact_failed`** is the terminal counterpart for telemetry.

## Sources, execution mode and limitations

**Execution mode:** **Run** — real sessions with the chapter's `message_end` handler, driven by a scripted model that returns an overflow error.

**Evidence scope:** `in_process_runtime`. The classification, recovery and context_edit are Pi's.

### What was read or run

- **Ran** `drivers/ch29-failures.ts`: three real sessions (normalised overflow, no handler, rate limit with 'length').
- **Ran** `ch29-failures/overflow.test.ts` (3 tests).
- **Read** `examples/evidence.json`, chapter 29 rows (3 claims).

### Limitations

- **Retry timings and a provider's transient-error classification are not verified** (`ch29-lim1`).
- **The substring is a stand-in** for a real provider's message — the handler matches `model is too long`.
- **One compact-and-retry attempt**, not a loop, is Pi's documented recovery limit.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
